# 第二课：Tokenizer、Embedding 与位置机制

通过 BPE 算法、真实 ByteLevel 配置、自动求导与旋转运算，解释模型输入。
先按 README 安装依赖，使用项目 Python 内核。本地配置已保存，不下载预训练模型。
课堂主讲标注的核心语句；辅助函数的完整实现可在 code/ 中查看。

In [1]:
from pathlib import Path
from collections import Counter
import sys
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

ROOT = next(p for p in [Path.cwd(), Path.cwd() / "class2", Path.cwd().parent]
            if (p / "code/bpe_core.py").exists())
sys.path.insert(0, str(ROOT / "code"))
from bpe_core import merge_pair
from runtime import load_tokenizer, make_batch
from embedding_train import bigram_batch
torch.set_num_threads(1)

## 1. BPE 训练：频次怎样影响每轮选择？

对每个当前相邻 pair 累加 `句子频次 × 出现次数`，选最高频，再重新统计。
这里是可读的字符级算法，负责解释规则；真实 ByteLevel 库在后面另训练。

In [2]:
def train_bpe(corpus, num_merges):
    """corpus: {文本:出现次数}；同频时按首次出现顺序选择。"""
    pieces = {text: list(text) for text in corpus}
    merges, history = [], []
    for step in range(num_merges):
        counts = Counter()
        for text, parts in pieces.items():
            for pair in zip(parts, parts[1:]):
                counts[pair] += corpus[text]
        if not counts:
            break
        pair = max(counts, key=counts.get)
        merges.append(pair)
        history.append({"rank": step, "pair": pair, "count": counts[pair],
                        "top_pairs": counts.most_common(3)})
        pieces = {text: merge_pair(parts, pair) for text, parts in pieces.items()}
    return merges, history

corpus = {"我喜欢喝牛奶": 5, "你喜欢喝牛奶": 3, "我喜欢喝酒": 2}
merges, history = train_bpe(corpus, 4)
for row in history:
    print(row)

{'rank': 0, 'pair': ('喜', '欢'), 'count': 10, 'top_pairs': [(('喜', '欢'), 10), (('欢', '喝'), 10), (('喝', '牛'), 8)]}
{'rank': 1, 'pair': ('喜欢', '喝'), 'count': 10, 'top_pairs': [(('喜欢', '喝'), 10), (('喝', '牛'), 8), (('牛', '奶'), 8)]}
{'rank': 2, 'pair': ('喜欢喝', '牛'), 'count': 8, 'top_pairs': [(('喜欢喝', '牛'), 8), (('牛', '奶'), 8), (('我', '喜欢喝'), 7)]}
{'rank': 3, 'pair': ('喜欢喝牛', '奶'), 'count': 8, 'top_pairs': [(('喜欢喝牛', '奶'), 8), (('我', '喜欢喝牛'), 5), (('你', '喜欢喝牛'), 3)]}


## 2. BPE 编码：rank 优先，而非最长匹配

重叠候选 `ab`、`bc` 谁先合并？当前候选的最小 rank 决定顺序。
`bc×6、ab×5、abc×1` 会学到 `bc` 优先于 `ab`，编码 `abc` 得到 `a/bc`。

In [3]:
def encode_bpe(text, merges):
    """每轮在当前相邻 pair 中选最小 rank，再合并；不按 token ID 或长度排序。"""
    ranks = {pair: rank for rank, pair in enumerate(merges)}
    parts, trace = list(text), []
    while len(parts) > 1:
        candidates = [(ranks[pair], pair) for pair in zip(parts, parts[1:])
                      if pair in ranks]
        if not candidates:
            break
        rank, pair = min(candidates, key=lambda item: item[0])
        before = parts[:]
        parts = merge_pair(parts, pair)
        trace.append({"rank": rank, "pair": pair, "before": before, "after": parts[:]})
    return parts, trace

conflict_merges, _ = train_bpe({"bc": 6, "ab": 5, "abc": 1}, 2)
print("规则：", conflict_merges)
print("rank 编码：", encode_bpe("abc", conflict_merges))

规则： [('b', 'c'), ('a', 'b')]
rank 编码： (['a', 'bc'], [{'rank': 0, 'pair': ('b', 'c'), 'before': ['a', 'b', 'c'], 'after': ['a', 'bc']}])


## 3. 用真实库训练并保存完整配置

`ByteLevel` 预切分、256个基础字节符号、4个特殊符号、BPE合并组成配置。
当前单元训练一个本地对象；后续实验使用仓库已保存的320/384配置，以固定ID。
小语料只用于机制观察，不是商用模型的 tokenizer。

In [4]:
SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]

def train(vocab_size):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), special_tokens=SPECIALS)
    corpus = (ROOT / "data/corpus.txt").read_text(encoding="utf-8").splitlines()
    tokenizer.train_from_iterator(corpus * 10, trainer=trainer)
    return tokenizer

fresh = train(320)
print("新训练词表大小：", fresh.get_vocab_size())
print("冻结配置词表大小：", load_tokenizer().get_vocab_size())




新训练词表大小： 320
冻结配置词表大小： 320


## 4. 对照词表规模、空格、标点与未见字节

观察内部 token 字符串、ID、offsets 和整串回解。ByteLevel token 不一定能单独解成完整汉字。
`Hello world!` 在本次320/384配置下分别为9/5项；并非每个文本都会缩短。

In [5]:
for size in (320, 384):
    tokenizer = load_tokenizer(size)
    for text in ["我喜欢喝牛奶。", "Hello world!", "Hello, world!", "未见过的字🦄"]:
        enc = tokenizer.encode(text)
        print(size, repr(text), len(enc.ids), enc.tokens, enc.ids)
        print("offsets / 回解相同：", enc.offsets, tokenizer.decode(enc.ids) == text)
tokenizer = load_tokenizer()
print("预切分：", tokenizer.pre_tokenizer.pre_tokenize_str("Hello, world!"))

320 '我喜欢喝牛奶。' 3 ['æĪĳåĸľæ¬¢åĸĿ', 'çīĽå¥¶', 'ãĢĤ'] [300, 304, 262]
offsets / 回解相同： [(0, 4), (4, 6), (6, 7)] True
320 'Hello world!' 9 ['H', 'el', 'l', 'o', 'Ġw', 'or', 'l', 'd', '!'] [43, 308, 79, 82, 295, 312, 79, 71, 4]
offsets / 回解相同： [(0, 1), (1, 3), (3, 4), (4, 5), (5, 7), (7, 9), (9, 10), (10, 11), (11, 12)] True
320 'Hello, world!' 10 ['H', 'el', 'l', 'o', ',', 'Ġw', 'or', 'l', 'd', '!'] [43, 308, 79, 82, 15, 295, 312, 79, 71, 4]
offsets / 回解相同： [(0, 1), (1, 3), (3, 4), (4, 5), (5, 6), (6, 8), (8, 10), (10, 11), (11, 12), (12, 13)] True
320 '未见过的字🦄' 19 ['æ', 'ľ', 'ª', 'è', '§', 'ģ', 'è', '¿', 'ĩ', 'ç', 'ļ', 'Ħ', 'å', 'Ń', 'Ĺ', 'ð', 'Ł', '¦', 'Ħ'] [166, 254, 107, 168, 104, 227, 168, 127, 233, 167, 252, 230, 165, 259, 249, 176, 257, 103, 230]
offsets / 回解相同： [(0, 1), (0, 1), (0, 1), (1, 2), (1, 2), (1, 2), (2, 3), (2, 3), (2, 3), (3, 4), (3, 4), (3, 4), (4, 5), (4, 5), (4, 5), (5, 6), (5, 6), (5, 6), (5, 6)] True
384 '我喜欢喝牛奶。' 3 ['æĪĳåĸľæ¬¢åĸĿ', 'çīĽå¥¶', 'ãĢĤ'] [300, 304, 262]
off

## 5. batch 查表与 one-hot 前向等价

IDs `[B,T]` → 表E `[V,D]` → 向量 `[B,T,D]`。
这里只为了验证构造one-hot；实际训练不需要分配如此大的中间张量。

In [6]:
torch.manual_seed(7)
ids, valid = make_batch(tokenizer, ["我喜欢喝牛奶。", "我喜欢喝酒。"])
pad = tokenizer.token_to_id("<pad>")
embedding = nn.Embedding(tokenizer.get_vocab_size(), 8, padding_idx=pad)
vectors = embedding(ids)
one_hot = F.one_hot(ids, tokenizer.get_vocab_size()).to(vectors.dtype)
print("IDs / E / X：", list(ids.shape), list(embedding.weight.shape), list(vectors.shape))
print("前向等价：", torch.allclose(vectors, one_hot @ embedding.weight))
print("valid mask：", valid.tolist())

IDs / E / X： [2, 6] [320, 8] [2, 6, 8]
前向等价： True
valid mask： [[True, True, True, True, True, False], [True, True, True, True, True, True]]


## 6. gather 反传：同一行的梯度累加

对 `[a,a,b,pad]` 的输出求和，上游每维都是1。a行收到两次贡献，b一次，pad的lookup梯度为0。
这只是诊断loss。`padding_idx`不会自动屏蔽Attention或loss；权重绑定时还有输出梯度路径。

In [7]:
a, b = tokenizer.encode("我喜欢喝牛奶。").ids[:2]
embedding.zero_grad()
embedding(torch.tensor([[a, a, b, pad]])).sum().backward()
print("a行梯度：", embedding.weight.grad[a].tolist())
print("b行梯度：", embedding.weight.grad[b].tolist())
print("pad行梯度：", embedding.weight.grad[pad].tolist())

a行梯度： [2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0]
b行梯度： [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
pad行梯度： [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 7. 单位置交叉熵梯度：手算与 autograd 对齐

`z=eW^T`，`p=softmax(z)`，`∂L/∂e=(p−one_hot(y))W`。本例一个位置，不含batch平均因子。
将手算结果与自动求导逐项比较。

In [8]:
torch.manual_seed(7)
e = torch.randn(1, 4, dtype=torch.float64, requires_grad=True)
W = torch.randn(6, 4, dtype=torch.float64)
target = torch.tensor([2])
logits = e @ W.T
F.cross_entropy(logits, target).backward()
manual = (logits.detach().softmax(-1) - F.one_hot(target, 6)) @ W
print("autograd：", e.grad.tolist())
print("手算：", manual.tolist())
torch.testing.assert_close(e.grad, manual)

autograd： [[0.1561607187493172, -0.5626540431463162, 0.11168024702582799, 0.2704132746653008]]
手算： [[0.15616071874931717, -0.5626540431463162, 0.11168024702582795, 0.2704132746653008]]


## 8. 真实下一 token 损失更新 embedding

七句文本构成57对相邻token。`Embedding → Linear → CrossEntropy` 是bigram基线：每次只看一个token。
输出原始logits给交叉熵。没有验证集；观察训练拟合，不把它称为Transformer。

In [9]:
torch.manual_seed(7)
x, y = bigram_batch(tokenizer)
token_embedding = nn.Embedding(tokenizer.get_vocab_size(), 16)
head = nn.Linear(16, tokenizer.get_vocab_size())
optimizer = torch.optim.Adam(list(token_embedding.parameters()) + list(head.parameters()), lr=0.05)
before = token_embedding.weight.detach().clone()
for step in range(121):
    loss = F.cross_entropy(head(token_embedding(x)), y)
    if step % 30 == 0:
        print(step, round(float(loss.detach()), 4))
    if step == 120:
        break
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print("改变表行数：", int(((token_embedding.weight.detach()-before).norm(dim=1) > 0).sum()))

0 5.7565
30 0.3607
60 0.3564
90 0.356
120 0.3559
改变表行数： 26


## 9. 无位置、无遮罩的注意力：置换等变

重排输入后输出也对应重排。给重新排列的token使用新的当前位置编码，则通常不再满足这一等式。
这里用Q=K=V=X做最小验证，因果mask不在该等变结论的假设中。

In [10]:
from position_mechanics import demo as check_permutation
check_permutation()

无位置：Attn(PX) == P Attn(X)： True
重排 token、保留当前位置索引后仍等变吗： False
此实验不加因果 mask；不能把该等变结论直接套到带 mask 的场景。


## 10. 向量化正弦位置编码

`[T,1] × [1,D/2] → [T,D/2]` 的广播产生各位置、各频率的角度。
相邻两维分别放sin/cos。固定偏移可由一组二维旋转表达。

In [11]:
def sinusoidal(positions, width):
    if width <= 0 or width % 2:
        raise ValueError("width 必须为正偶数")
    positions = torch.as_tensor(positions, dtype=torch.float64)
    frequencies = 10000.0 ** (-torch.arange(0, width, 2, dtype=torch.float64) / width)
    angles = positions[:, None] * frequencies[None, :]
    result = torch.empty(len(positions), width, dtype=torch.float64)
    result[:, 0::2], result[:, 1::2] = angles.sin(), angles.cos()
    return result

print(sinusoidal(torch.arange(3), 4).round(decimals=4))

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  1.0000],
        [ 0.9093, -0.4161,  0.0200,  0.9998]], dtype=torch.float64)


## 11. RoPE：成对旋转与相对位移

q′=R(mθ)q，k′=R(nθ)k，所以q′^Tk′=q^TR((n−m)θ)k。
实现采用相邻维度配对，输入可为`[...,T,D]`，广播到batch和head。整体平移位置测试固定未旋转的Q/K内容。

In [12]:
def apply_rope(x, positions):
    width = x.shape[-1]
    positions = torch.as_tensor(positions, device=x.device, dtype=x.dtype)
    if width <= 0 or width % 2 or positions.ndim != 1 or len(positions) != x.shape[-2]:
        raise ValueError("要求 x=[...,T,D]、positions=[T]，D为正偶数")
    frequencies = 10000.0 ** (-torch.arange(0, width, 2, device=x.device, dtype=x.dtype) / width)
    angles = positions[:, None] * frequencies[None, :]
    even, odd = x[..., 0::2], x[..., 1::2]
    result = torch.empty_like(x)
    result[..., 0::2] = even * angles.cos() - odd * angles.sin()
    result[..., 1::2] = even * angles.sin() + odd * angles.cos()
    return result

torch.manual_seed(7)
q, k = torch.randn(2, 3, 4, dtype=torch.float64)
p = torch.arange(3)
score = apply_rope(q, p) @ apply_rope(k, p).T
shifted = apply_rope(q, p+5) @ apply_rope(k, p+5).T
print("整体平移后最大差异：", float((score-shifted).abs().max()))
print("范数保持：", torch.allclose(q.norm(dim=-1), apply_rope(q, p).norm(dim=-1)))

整体平移后最大差异： 1.3322676295501878e-15
范数保持： True


## 12. 接口整合：真实ID、多头Q/K、RoPE、两种mask

打开`code/input_pipeline.py`，核对每步shape。此分支直接在Q/K使用RoPE，不叠加相加式绝对位置。
参数是随机初始化；脚本不包含完整Block、语言模型训练或生成循环。

In [13]:
from input_pipeline import demo as input_pipeline
pipeline_result = input_pipeline()

ID / embedding： [2, 6] [2, 6, 8]
Q/K： [2, 2, 6, 4] scores： [2, 2, 6, 6]
输出： [2, 6, 8]
未来与 padding key 权重均为0： True


## 复习与作业

解释：rank与最长前缀为何不同？重复token梯度为何累加？绑定输入输出权重后谁还会更新？
RoPE与因果mask各控制什么？详见 [练习](exercises.md) 与 [讲义](student_handout.md)。